# 🐼 Pandas Mastery: A Professional Engineer's Guide

> **Audience:** Mid–Senior Software Engineers new to Pandas  
> **Goal:** Production-level proficiency from scratch  
> **Duration:** ~6–8 hours of focused study

---

## 📚 Table of Contents

### 🟢 BEGINNER
1. [Setup & Installation](#1-setup)
2. [Series – The 1D Building Block](#2-series)
3. [DataFrame – The Core Data Structure](#3-dataframe)
4. [Reading & Writing Data (I/O)](#4-io)
5. [Inspecting & Exploring Data](#5-inspection)
6. [Selecting & Indexing Data](#6-indexing)
7. [Filtering Data (Boolean Masking)](#7-filtering)
8. [Adding, Updating & Dropping Columns/Rows](#8-mutations)

### 🟡 INTERMEDIATE
9. [Handling Missing Data](#9-missing)
10. [Data Type Casting & Conversion](#10-dtypes)
11. [String Operations (.str Accessor)](#11-strings)
12. [DateTime Operations (.dt Accessor)](#12-datetime)
13. [GroupBy & Aggregations](#13-groupby)
14. [Merging, Joining & Concatenating](#14-merging)
15. [Pivot Tables & Reshaping](#15-pivot)
16. [Apply, Map & Vectorization](#16-apply)
17. [Sorting & Ranking](#17-sorting)

### 🔴 ADVANCED
18. [MultiIndex & Advanced Indexing](#18-multiindex)
19. [Window Functions (Rolling, Expanding, EWM)](#19-windows)
20. [Performance Optimization & Memory](#20-performance)
21. [Method Chaining & Pipelines](#21-chaining)
22. [Categorical Data](#22-categorical)
23. [Common Pitfalls & Debugging](#23-pitfalls)
24. [Interview-Critical Concepts Cheatsheet](#24-interview)

### 🏆 FINAL PROJECT
25. [Mini Project: E-Commerce Sales Analysis Pipeline](#25-project)

---

---
# 🟢 BEGINNER
---

<a id='1-setup'></a>
## 1. Setup & Installation

Pandas is pre-installed in Google Colab. In production, pin versions — pandas 1.x→2.x had breaking changes (`append()` removal, Copy-on-Write semantics). Always check compatibility when upgrading.

In [ ]:
import pandas as pd
import numpy as np
import time, io, os

print(f'Pandas version : {pd.__version__}')
print(f'NumPy version  : {np.__version__}')

# Useful display settings for development
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)
print('\n✅ Setup complete!')

<a id='2-series'></a>
## 2. Series – The 1D Building Block

A `Series` is a **labeled 1D array**. Think of it as an ordered dictionary. Every DataFrame column is a Series. Unlike a NumPy array, a Series carries:
- An explicit **index** (not just positional integers)
- A single **dtype** and a **name** attribute
- **NaN-aware** operations

In [ ]:
prices = pd.Series(
    [120.5, 85.0, 200.0, np.nan, 150.75],
    index=['AAPL', 'GOOG', 'MSFT', 'AMZN', 'META'],
    name='closing_price'
)
print('=== Series ===')
print(prices)
print(f'\ndtype  : {prices.dtype}')
print(f'name   : {prices.name}')

# Label-based vs positional access
print(f"\nBy label  prices['AAPL']  = {prices['AAPL']}")
print(f'By position prices.iloc[0] = {prices.iloc[0]}')

# Vectorized operations
print('\n=== Prices after 10% gain ===')
print(prices * 1.10)

# Boolean masking
print('\n=== Stocks above $140 ===')
print(prices[prices > 140])

In [ ]:
print(f'Sum     : {prices.sum()}')
print(f'Mean    : {prices.mean():.2f}')
print(f'Median  : {prices.median():.2f}')
print(f'NaN count : {prices.isna().sum()}')

status = pd.Series(['active','inactive','active','banned','active','inactive'])
print('\n=== value_counts() ===')
print(status.value_counts())
print('\nNormalized:')
print(status.value_counts(normalize=True).round(2))

> 💡 **Interview Tip:** `value_counts()` is O(n) using a hash map — faster than `groupby().count()` for single-column frequency tables. For large cardinality columns in groupby, consider `category` dtype first.

### 🏋️ Exercise 2
Given `daily_users` below:
1. Find the day with maximum users (`idxmax`)
2. Calculate day-over-day % change (`pct_change`)
3. Filter days where users exceeded the mean

In [ ]:
daily_users = pd.Series(
    [1200, 1350, 980, 1500, 1620, 1100, 1750],
    index=pd.date_range('2024-01-01', periods=7, freq='D'),
    name='dau'
)
# --- YOUR CODE HERE ---
# 1. Day with max users

# 2. Day-over-day % change

# 3. Days above mean


In [ ]:
# ✅ SOLUTION
print('1. Max day:', daily_users.idxmax(), '->', daily_users.max(), 'users')
print('\n2. Day-over-day % change:')
print(daily_users.pct_change().mul(100).round(1).to_string())
print('\n3. Days above mean:')
print(daily_users[daily_users > daily_users.mean()])

<a id='3-dataframe'></a>
## 3. DataFrame – The Core Data Structure

A DataFrame is a **2D labeled table** — a dict of Series sharing the same index. Internally data is stored **column-major** (each column is a contiguous NumPy array). This is why column operations are fast and row-wise operations are expensive.

In [ ]:
# From list of dicts (most common in practice)
employees = pd.DataFrame([
    {'id':101,'name':'Alice',  'dept':'Engineering','salary':120000,'yoe':5},
    {'id':102,'name':'Bob',    'dept':'Marketing',  'salary':95000, 'yoe':3},
    {'id':103,'name':'Charlie','dept':'Engineering','salary':140000,'yoe':8},
    {'id':104,'name':'Diana',  'dept':'Data',       'salary':130000,'yoe':6},
    {'id':105,'name':'Eve',    'dept':'Data',       'salary':115000,'yoe':4},
    {'id':106,'name':'Frank',  'dept':'Marketing',  'salary':88000, 'yoe':2},
    {'id':107,'name':'Grace',  'dept':'Engineering','salary':160000,'yoe':10},
])
print(employees)
print(f'\nShape: {employees.shape}  ({employees.shape[0]} rows x {employees.shape[1]} cols)')
print(f'\nKey attributes:')
print(f'  columns : {employees.columns.tolist()}')
print(f'  dtypes  :')
print(employees.dtypes)

In [ ]:
# From dict of lists
stock_data = pd.DataFrame({
    'ticker': ['AAPL','GOOG','MSFT'],
    'price':  [189.5, 141.2, 420.0],
    'volume': [55_000_000, 20_000_000, 35_000_000],
})

# From NumPy array with custom index
matrix_df = pd.DataFrame(
    np.random.randint(0, 100, size=(3, 4)),
    columns=['q1','q2','q3','q4'],
    index=['product_A','product_B','product_C']
)
print('=== Stock Data ==='); print(stock_data)
print('\n=== Matrix DF ==='); print(matrix_df)

<a id='4-io'></a>
## 4. Reading & Writing Data (I/O)

In production you ingest from CSV, Parquet, databases, and APIs. **Parquet is preferred** for data engineering — columnar, compressed, schema-preserving.

In [ ]:
employees.to_csv('/tmp/employees.csv', index=False)
employees.to_json('/tmp/employees.json', orient='records')
employees.to_parquet('/tmp/employees.parquet')

print('File sizes:')
for f in ['/tmp/employees.csv','/tmp/employees.json','/tmp/employees.parquet']:
    print(f'  {f:42s} {os.path.getsize(f):>8,} bytes')

In [ ]:
df_csv     = pd.read_csv('/tmp/employees.csv')
df_parquet = pd.read_parquet('/tmp/employees.parquet')

print('=== read_csv dtypes (inferred) ===')
print(df_csv.dtypes)
print('\n=== read_parquet dtypes (preserved!) ===')
print(df_parquet.dtypes)

In [ ]:
csv_data = """# Data export v2.1
user_id,email,signup_date,plan,mrr
U001,alice@co.com,2023-01-15,pro,99.0
U002,bob@co.com,2023-03-22,free,0.0
U003,charlie@co.com,2023-05-01,enterprise,499.0
"""
users = pd.read_csv(
    io.StringIO(csv_data),
    skiprows=1,                               # skip comment
    parse_dates=['signup_date'],              # parse dates
    dtype={'user_id': str, 'mrr': float},     # explicit dtypes
    na_values=['','N/A','null'],              # NA tokens
)
print(users)
print('\ndtypes:'); print(users.dtypes)

> 💡 **Interview Tip:** `read_csv` is single-threaded. For files > 1 GB, use `chunksize` to iterate, or switch to **Polars/Dask** for parallel reads. Always specify `dtype=` — without it, pandas infers types by scanning the full file, doubling memory usage.

<a id='5-inspection'></a>
## 5. Inspecting & Exploring Data

Before any transformation, understand the shape, quality, and distribution of your data. These are the **first commands** you run on every new dataset.

In [ ]:
np.random.seed(42)
n = 500
sales = pd.DataFrame({
    'order_id':    range(1001, 1001+n),
    'customer_id': np.random.randint(1, 150, n),
    'product':     np.random.choice(['Laptop','Phone','Tablet','Monitor','Keyboard'], n),
    'region':      np.random.choice(['North','South','East','West'], n),
    'quantity':    np.random.randint(1, 10, n),
    'unit_price':  np.random.uniform(50, 1500, n).round(2),
    'order_date':  pd.date_range('2023-01-01', periods=n, freq='12h'),
    'discount':    np.random.choice([0, 0.05, 0.10, 0.15, np.nan], n),
})
sales['revenue'] = (sales['quantity'] * sales['unit_price'] * (1 - sales['discount'].fillna(0))).round(2)

print('--- .head() ---'); print(sales.head(3))
print(f'\nShape: {sales.shape}')
print('\n--- .info() ---'); sales.info()

In [ ]:
print('--- .describe() numeric ---'); print(sales.describe())
print('\n--- .describe() categorical ---')
print(sales[['product','region']].describe())

In [ ]:
print('=== Null Audit ===')
null_summary = pd.DataFrame({
    'null_count':   sales.isna().sum(),
    'null_pct':     (sales.isna().mean()*100).round(2),
    'dtype':        sales.dtypes,
    'unique_count': sales.nunique()
}).sort_values('null_pct', ascending=False)
print(null_summary)

<a id='6-indexing'></a>
## 6. Selecting & Indexing Data

| Method | Use case | Notes |
|--------|----------|-------|
| `df['col']` | Single column | Returns Series |
| `df[['a','b']]` | Multiple columns | Returns DataFrame |
| `df.loc[label]` | Label-based | Inclusive on both ends |
| `df.iloc[int]` | Position-based | Exclusive end (Python-style) |
| `df.at[r,c]` | Single scalar by label | Faster than .loc |
| `df.iat[r,c]` | Single scalar by position | Fastest scalar access |

In [ ]:
df = employees.copy().set_index('id')
print('=== id as index ===')
print(df)

print('\n--- .loc[101] single row ---')
print(df.loc[101])

print("\n--- .loc[101:104, ['name','salary']] ---")
print(df.loc[101:104, ['name','salary']])

In [ ]:
print('--- .iloc[0:3, 0:3] ---')
print(df.iloc[0:3, 0:3])

print('\n--- .iloc[-2:] last 2 rows ---')
print(df.iloc[-2:])

print(f"\n.at[102,'salary']  = {df.at[102,'salary']}")
print(f'.iat[2, 2]          = {df.iat[2, 2]}')

> 💡 **Interview Tip:** `.loc` is **inclusive** on both ends; Python slices are exclusive on the right. This trips up many engineers. Also, `df['col']` is `df.__getitem__('col')` — useful when building dynamic column access utilities.

<a id='7-filtering'></a>
## 7. Filtering Data (Boolean Masking)

Boolean masking is the idiomatic Pandas way to filter. Under the hood it applies a boolean NumPy array as a row selector — O(n), no Python loops.

In [ ]:
df = employees.copy()

print('--- Engineers ---')
print(df[df['dept'] == 'Engineering'])

# Multiple conditions — use & | ~ (NOT and/or/not)
print('\n--- Salary > $120k AND Engineering/Data ---')
mask = (df['salary'] > 120000) & (df['dept'].isin(['Engineering','Data']))
print(df[mask])

# .query() — readable for complex filters
print('\n--- .query() ---')
print(df.query("salary > 100000 and yoe >= 5 and dept != 'Marketing'"))

In [ ]:
print('--- .isin() ---')
print(df[df['dept'].isin(['Engineering','Data'])])

print('\n--- .between() 90k-130k ---')
print(df[df['salary'].between(90_000, 130_000)])

print('\n--- ~ NOT operator ---')
print(df[~df['dept'].isin(['Marketing'])])

### 🏋️ Exercise 7
Using `sales`:
1. Filter North/East region orders with revenue > 2000
2. Use `.query()` for Phone/Tablet orders, quantity >= 5
3. Find top 5 highest revenue orders

In [ ]:
# --- YOUR CODE HERE ---
# 1.

# 2.

# 3.


In [ ]:
# ✅ SOLUTION
r1 = sales[sales['region'].isin(['North','East']) & (sales['revenue'] > 2000)]
print(f'1. Count: {len(r1)}')

r2 = sales.query("product in ['Phone','Tablet'] and quantity >= 5")
print(f'\n2. Count: {len(r2)}')
print(r2[['product','quantity','revenue']].head())

print('\n3. Top 5:')
print(sales.nlargest(5,'revenue')[['order_id','product','region','revenue']])

<a id='8-mutations'></a>
## 8. Adding, Updating & Dropping Columns/Rows

Key design principle: prefer **non-mutating** operations via `.assign()` and use `.copy()` to avoid `SettingWithCopyWarning` (formalized as Copy-on-Write in Pandas 2.x).

In [ ]:
df = employees.copy()

# Adding columns
df['annual_bonus'] = df['salary'] * 0.10
df['senior']       = df['yoe'] >= 6
df['salary_band']  = pd.cut(df['salary'],
                            bins=[0, 100_000, 130_000, float('inf')],
                            labels=['junior','mid','senior'])

print(df[['name','salary','yoe','annual_bonus','senior','salary_band']])

In [ ]:
# Updating values with .loc
df.loc[df['dept']=='Engineering','salary'] *= 1.05
print('--- After 5% engineering raise ---')
print(df[df['dept']=='Engineering'][['name','salary']])

# Dropping
df_clean = df.drop(columns=['annual_bonus','senior']).drop(index=[0,1])
print('\n--- After drops ---'); print(df_clean.head())

# .assign() — immutable, chainable (PREFERRED in pipelines)
df2 = (
    employees.copy()
    .assign(
        salary_usd = lambda x: x['salary'],
        salary_eur = lambda x: x['salary'] * 0.92,
        level      = lambda x: pd.cut(x['yoe'], bins=[0,3,7,100], labels=['L1','L2','L3'])
    )
)
print('\n--- .assign() result ---')
print(df2[['name','salary_usd','salary_eur','level']])

---
# 🟡 INTERMEDIATE
---

<a id='9-missing'></a>
## 9. Handling Missing Data

Real-world data is always dirty. Pandas represents missing data as `NaN` (float), `None` (Python), `NaT` (datetime), or `pd.NA` (extension arrays). Which one appears affects type inference.

In [ ]:
df_miss = pd.DataFrame({
    'user_id': range(1,11),
    'age':    [25, np.nan, 32, np.nan, 28, 35, np.nan, 41, 29, 33],
    'score':  [88, 72, np.nan, 95, np.nan, 60, 78, np.nan, 85, 90],
    'city':   ['NY','LA',None,'SF','NY',None,'LA','SF',None,'NY'],
    'signup': pd.to_datetime(['2023-01','2023-02',None,'2023-04',
                              '2023-05','2023-06',None,'2023-08','2023-09','2023-10'])
})
print(df_miss)
print('\nNull counts:'); print(df_miss.isna().sum())

In [ ]:
# Drop (use thresh to require minimum non-null values per row)
print('--- dropna(thresh=3) ---')
print(df_miss.dropna(thresh=3))

# Fill with constant / statistic
df_filled = df_miss.copy()
df_filled['city'] = df_filled['city'].fillna('Unknown')
df_filled['age']  = df_filled['age'].fillna(df_filled['age'].median())

# Forward fill (time-series: carry last valid value forward)
df_ffill = df_miss.copy()
df_ffill['score'] = df_ffill['score'].ffill()

# Interpolation
df_interp = df_miss.copy()
df_interp['score'] = df_interp['score'].interpolate(method='linear')

print('\n--- Comparison: original vs ffill vs interpolate ---')
print(pd.DataFrame({
    'original':    df_miss['score'],
    'ffill':       df_ffill['score'],
    'interpolated':df_interp['score'],
}))

> 💡 **Interview Tip:** In production ML pipelines, imputation strategy from training data must be *applied* (not re-fit) on test data to prevent leakage. Use sklearn's `SimpleImputer` fitted on train, not raw pandas `.fillna()` on the whole dataset.

<a id='10-dtypes'></a>
## 10. Data Type Casting & Conversion

Correct dtypes are crucial. A column of integers stored as `object` will cause arithmetic to fail silently or with opaque errors.

In [ ]:
raw = pd.DataFrame({
    'user_id':    ['001','002','003','004'],
    'revenue':    ['$1,200.50','$850.00','$3,400.00','$999.99'],
    'is_premium': ['True','False','True','True'],
    'signup':     ['Jan 15 2023','2023-03-22','2023/05/01','15-07-2023'],
})
print('=== Raw (all objects) ==='); print(raw); print(raw.dtypes)

In [ ]:
clean = raw.copy()
clean['revenue']    = clean['revenue'].str.replace('[$,]','',regex=True).astype(float)
clean['is_premium'] = clean['is_premium'].map({'True':True,'False':False})
clean['signup']     = pd.to_datetime(clean['signup'], infer_datetime_format=True)
clean['user_id']    = clean['user_id'].astype(int)

print('=== Cleaned ==='); print(clean)
print('\ndtypes:'); print(clean.dtypes)

# Safe cast — bad values become NaN instead of crashing
messy = pd.Series(['100','200','n/a','three hundred','400'])
print('\n=== Safe numeric cast ===')
print(pd.to_numeric(messy, errors='coerce'))

<a id='11-strings'></a>
## 11. String Operations (`.str` Accessor)

The `.str` accessor vectorizes Python string methods across a Series. Always prefer `.str` methods over `.apply(lambda x: x.method())` — they're faster and null-safe.

In [ ]:
users_str = pd.DataFrame({
    'full_name': ['  Alice Johnson ','bob SMITH',"Charlie O'Brien",'diana-lee'],
    'email':     ['alice@COMPANY.com','bob@gmail.com','charlie@company.com','diana@yahoo.com'],
})

users_clean = users_str.copy()
users_clean['full_name']   = users_str['full_name'].str.strip().str.title()
users_clean['email']       = users_str['email'].str.lower()
users_clean['domain']      = users_str['email'].str.lower().str.split('@').str[1]
users_clean['is_corporate']= users_str['email'].str.lower().str.contains('@company\\.com')

print('=== Cleaned ==='); print(users_clean)

In [ ]:
# str.extract: regex named groups — great for log parsing
log_lines = pd.Series([
    '2024-01-15 ERROR user_id=42 msg=Login failed',
    '2024-01-16 INFO  user_id=7  msg=Password changed',
    '2024-01-16 WARN  user_id=99 msg=Rate limit reached',
])

parsed = log_lines.str.extract(
    r'(?P<date>\d{4}-\d{2}-\d{2}) (?P<level>\w+)\s+user_id=(?P<user_id>\d+) msg=(?P<message>.+)'
)
print('=== Parsed log entries ==='); print(parsed)

<a id='12-datetime'></a>
## 12. DateTime Operations (`.dt` Accessor)

Time-series data is in virtually every production system. The `.dt` accessor gives vectorized datetime components and arithmetic.

In [ ]:
np.random.seed(1)
dates = pd.date_range('2023-01-01','2023-12-31',freq='6h')
txn = pd.DataFrame({
    'ts':     np.random.choice(dates,300),
    'amount': np.random.exponential(200,300).round(2),
    'type':   np.random.choice(['debit','credit'],300),
}).sort_values('ts').reset_index(drop=True)

txn['year']        = txn['ts'].dt.year
txn['month']       = txn['ts'].dt.month
txn['hour']        = txn['ts'].dt.hour
txn['day_of_week'] = txn['ts'].dt.day_name()
txn['is_weekend']  = txn['ts'].dt.dayofweek >= 5
txn['quarter']     = txn['ts'].dt.quarter

print(txn.head(3))

In [ ]:
# Timezone handling
txn['ts_utc'] = txn['ts'].dt.tz_localize('UTC')
txn['ts_ist'] = txn['ts_utc'].dt.tz_convert('Asia/Kolkata')
print('=== TZ comparison ==='); print(txn[['ts','ts_utc','ts_ist']].head(3))

# Resample: powerful time-series aggregation
monthly = txn.set_index('ts')['amount'].resample('ME').agg(['sum','count','mean']).round(2)
monthly.columns = ['total_amount','transaction_count','avg_amount']
print('\n=== Monthly Aggregation via resample ===')
print(monthly)

> 💡 **Interview Tip:** Always store timestamps in UTC in your database. Convert to local time only at display layer. Timezone-naive and timezone-aware timestamps cannot be compared or merged — causes `TypeError` in production pipelines. Very common subtle bug.

<a id='13-groupby'></a>
## 13. GroupBy & Aggregations

GroupBy implements **split-apply-combine**. Internally it uses a hash map to group row indices, then applies functions to each group. The three modes are:
- `.agg()` — reduces each group to a scalar
- `.transform()` — returns same-shape result aligned to original index
- `.filter()` — drops entire groups based on a predicate

In [ ]:
df = sales.copy()

print('=== Revenue by region ===')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

print('\n=== Multi-metric summary by product ===')
product_stats = df.groupby('product').agg(
    total_revenue = ('revenue',  'sum'),
    order_count   = ('order_id', 'count'),
    avg_revenue   = ('revenue',  'mean'),
    avg_qty       = ('quantity', 'mean'),
    max_revenue   = ('revenue',  'max'),
).round(2)
print(product_stats)

In [ ]:
# transform: add group stats back to original df (feature engineering)
df['region_total']  = df.groupby('region')['revenue'].transform('sum')
df['region_rank']   = df.groupby('region')['revenue'].rank(ascending=False, method='dense')
df['pct_of_region'] = (df['revenue']/df['region_total']*100).round(2)

print('=== With group-level stats (transform) ===')
print(df[['order_id','region','revenue','region_total','pct_of_region']].head(8))

In [ ]:
# filter: keep groups meeting a condition
high_rev = df.groupby('region').filter(lambda g: g['revenue'].sum() > 100_000)
print(f'Regions > $100k total: {high_rev["region"].unique().tolist()}')

# Custom agg function
print('\n=== Revenue range per product ===')
print(df.groupby('product')['revenue']
      .agg(lambda x: x.max()-x.min())
      .rename('range')
      .sort_values(ascending=False))

### 🏋️ Exercise 13
1. Find the product with the highest average revenue per order
2. Add `is_top_customer` = True if customer_id is in top 10% by total spend

In [ ]:
# --- YOUR CODE HERE ---


In [ ]:
# ✅ SOLUTION
print('1.', sales.groupby('product')['revenue'].mean().idxmax())

customer_total = sales.groupby('customer_id')['revenue'].sum()
threshold      = customer_total.quantile(0.90)
top_customers  = customer_total[customer_total >= threshold].index
s2 = sales.copy()
s2['is_top_customer'] = s2['customer_id'].isin(top_customers)
print(f'\n2. Top 10%: {len(top_customers)} customers')
print(s2['is_top_customer'].value_counts())

<a id='14-merging'></a>
## 14. Merging, Joining & Concatenating

Pandas merge = SQL JOIN.

| Pandas | SQL |
|--------|-----|
| `how='inner'` | `INNER JOIN` |
| `how='left'`  | `LEFT OUTER JOIN` |
| `how='outer'` | `FULL OUTER JOIN` |
| `how='cross'` | `CROSS JOIN` |

In [ ]:
orders_m = pd.DataFrame({
    'order_id':   [1,2,3,4,5],
    'customer_id':[10,20,10,30,40],
    'amount':     [250,500,125,750,300],
})
customers_m = pd.DataFrame({
    'customer_id':[10,20,30,50],  # 40 missing, 50 has no orders
    'name':       ['Alice','Bob','Charlie','Eve'],
    'plan':       ['pro','free','enterprise','pro'],
})

print('=== INNER ===')
print(pd.merge(orders_m, customers_m, on='customer_id', how='inner'))

print('\n=== LEFT ===')
print(pd.merge(orders_m, customers_m, on='customer_id', how='left'))

print('\n=== OUTER ===')
print(pd.merge(orders_m, customers_m, on='customer_id', how='outer'))

In [ ]:
q1 = pd.DataFrame({'month':['Jan','Feb','Mar'],'sales':[100,120,110]})
q2 = pd.DataFrame({'month':['Apr','May','Jun'],'sales':[130,115,140]})
print('=== Vertical concat (UNION ALL) ===')
print(pd.concat([q1,q2], ignore_index=True))

# merge_asof: time-based approximate join (financial / event data)
quotes = pd.DataFrame({
    'ts':    pd.to_datetime(['09:00','09:05','09:10','09:20'],format='%H:%M'),
    'price': [100.0, 101.5, 102.0, 101.0]
})
trades = pd.DataFrame({
    'ts':  pd.to_datetime(['09:03','09:08','09:18'],format='%H:%M'),
    'qty': [500, 300, 700]
})
print('\n=== merge_asof: match each trade to most recent quote ===')
print(pd.merge_asof(trades, quotes, on='ts'))

> 💡 **Interview Tip:** Always check row counts before/after merge. Unexpected M:N joins create a cartesian product explosion that is catastrophic in production. Use `validate='m:1'` or `validate='1:1'` to enforce join cardinality — pandas will raise if the assumption is violated.

<a id='15-pivot'></a>
## 15. Pivot Tables & Reshaping

Reshaping moves data between **wide** and **long** formats. Fundamental for reporting and charting library interoperability.

In [ ]:
df = sales.copy()
pivot = pd.pivot_table(
    df, values='revenue', index='region', columns='product',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
).round(0)
print('=== Revenue Pivot: Region x Product ===')
print(pivot)

In [ ]:
wide_df = pd.DataFrame({
    'user_id': [1,2,3],
    'jan_rev': [100,200,150],
    'feb_rev': [110,190,160],
    'mar_rev': [120,210,145],
})

# Wide → Long
long_df = pd.melt(wide_df, id_vars=['user_id'], var_name='month', value_name='revenue')
long_df['month'] = long_df['month'].str.replace('_rev','')
print('=== Wide -> Long (melt) ==='); print(long_df)

# Long → Wide
print('\n=== Long -> Wide (pivot) ===')
print(long_df.pivot(index='user_id', columns='month', values='revenue'))

<a id='16-apply'></a>
## 16. Apply, Map & Vectorization

Many engineers default to `.apply()` for everything — it's the slowest option because it falls back to Python loops.

**Speed ranking (fastest → slowest):**  
`NumPy ufuncs` > `Vectorized pandas ops` > `.str/.dt` > `map()` > `apply(axis=0)` > `apply(axis=1)` > `Python loops`

In [ ]:
df = sales.head(100).copy()

# apply() on a Series
def tier_label(rev):
    if rev < 500:    return 'bronze'
    elif rev < 2000: return 'silver'
    else:            return 'gold'

df['tier_apply'] = df['revenue'].apply(tier_label)

# np.select: vectorized, much faster
conditions = [df['revenue'] < 500, df['revenue'].between(500,2000)]
df['tier_vec'] = np.select(conditions, ['bronze','silver'], default='gold')

# pd.cut: cleanest for binning
df['tier_cut'] = pd.cut(df['revenue'], bins=[0,500,2000,float('inf')],
                         labels=['bronze','silver','gold'])

print('All three match:', (df['tier_apply'] == df['tier_vec']).all())
print(df[['revenue','tier_apply','tier_vec','tier_cut']].head(5))

In [ ]:
big = pd.DataFrame({'x': np.random.randn(500_000)})

t0 = time.time(); _ = big['x'].apply(lambda v: v**2 + 2*v + 1); t_apply = time.time()-t0
t0 = time.time(); _ = big['x']**2 + 2*big['x'] + 1;             t_vec   = time.time()-t0

print(f'apply()    : {t_apply:.3f}s')
print(f'vectorized : {t_vec:.4f}s')
print(f'Speedup    : {t_apply/t_vec:.0f}x faster with vectorization')

<a id='17-sorting'></a>
## 17. Sorting & Ranking

Full sort is O(n log n). For top-K, `.nlargest()` / `.nsmallest()` use a heap and are O(n) — significantly faster for large DataFrames.

In [ ]:
print('=== Top 5 by revenue ===')
print(sales.sort_values('revenue',ascending=False).head(5)[['order_id','product','region','revenue']])

print('\n=== Multi-column sort (region ASC, revenue DESC) ===')
print(sales.sort_values(['region','revenue'],ascending=[True,False]).head(8)[['region','product','revenue']])

# Ranking
df = employees.copy()
df['salary_rank'] = df['salary'].rank(ascending=False, method='dense').astype(int)
df['dept_rank']   = df.groupby('dept')['salary'].rank(ascending=False, method='dense').astype(int)
df['pct_rank']    = df['salary'].rank(pct=True).round(2)

print('\n=== Rankings ===')
print(df[['name','dept','salary','salary_rank','dept_rank','pct_rank']].sort_values('salary_rank'))

---
# 🔴 ADVANCED
---

<a id='18-multiindex'></a>
## 18. MultiIndex & Advanced Indexing

MultiIndex (hierarchical indexing) represents higher-dimensional data in a 2D structure. Common in: financial data (date × symbol), geo data (region × city), aggregation results.

In [ ]:
idx = pd.MultiIndex.from_product(
    [['2023-Q1','2023-Q2','2023-Q3'],['North','South','East']],
    names=['quarter','region']
)
np.random.seed(7)
mi_df = pd.DataFrame(
    {'revenue': np.random.randint(50000,200000,len(idx)),
     'orders':  np.random.randint(100,500,len(idx))},
    index=idx
)
print('=== MultiIndex DataFrame ==='); print(mi_df)

In [ ]:
print('--- All Q1 ---'); print(mi_df.loc['2023-Q1'])
print('\n--- (Q2, North) ---'); print(mi_df.loc[('2023-Q2','North')])
print('\n--- xs: all quarters, North only ---'); print(mi_df.xs('North',level='region'))
print('\n--- Unstack region as columns ---'); print(mi_df['revenue'].unstack('region'))
print('\n--- Sum by quarter ---'); print(mi_df.groupby(level='quarter').sum())

<a id='19-windows'></a>
## 19. Window Functions (Rolling, Expanding, EWM)

Essential for time series: moving averages, volatility, drawdown, momentum signals.

In [ ]:
np.random.seed(42)
dates  = pd.date_range('2023-01-01',periods=100,freq='B')
prices = pd.DataFrame({
    'date':  dates,
    'close': 100 + np.cumsum(np.random.randn(100)*2),
    'volume':np.random.randint(1_000_000,10_000_000,100)
})

prices['ma_5']    = prices['close'].rolling(window=5).mean()
prices['ma_20']   = prices['close'].rolling(window=20).mean()
prices['vol_10']  = prices['close'].rolling(window=10).std()
prices['cummax']  = prices['close'].expanding().max()
prices['drawdown']= prices['close']/prices['cummax'] - 1
prices['ema_12']  = prices['close'].ewm(span=12,adjust=False).mean()

print(prices[['date','close','ma_5','ma_20','ema_12','drawdown']].tail(8).round(2))

In [ ]:
# Rolling within groups
ms = pd.DataFrame({
    'date':   pd.date_range('2023-01-01',periods=20,freq='B').tolist()*2,
    'ticker': ['AAPL']*20 + ['GOOG']*20,
    'price':  np.concatenate([
        100+np.cumsum(np.random.randn(20)),
        200+np.cumsum(np.random.randn(20))
    ])
})
ms['ma5'] = ms.groupby('ticker')['price'].transform(lambda x: x.rolling(5).mean())
print('=== Per-ticker 5-day MA ===')
print(ms[ms['ticker']=='AAPL'].tail(5).round(2))

<a id='20-performance'></a>
## 20. Performance Optimization & Memory

In production, DataFrames can be 10 GB+ in RAM. These techniques reduce memory by 5–20x.

In [ ]:
np.random.seed(0)
big = pd.DataFrame({
    'id':      np.random.randint(0,10000,100_000),
    'score':   np.random.rand(100_000),
    'category':np.random.choice(['A','B','C','D'],100_000),
})
mem_before = big.memory_usage(deep=True).sum()/1024**2

opt = big.copy()
opt['id']       = pd.to_numeric(opt['id'], downcast='unsigned')  # int64->uint16
opt['score']    = opt['score'].astype('float32')                  # float64->float32
opt['category'] = opt['category'].astype('category')              # object->category

mem_after = opt.memory_usage(deep=True).sum()/1024**2
print(f'Memory before: {mem_before:.2f} MB')
print(f'Memory after : {mem_after:.2f} MB')
print(f'Reduction    : {(1-mem_after/mem_before)*100:.1f}%')
print('\nOptimized dtypes:'); print(opt.dtypes)

In [ ]:
# Process large files in chunks (streaming approach)
large_csv = pd.DataFrame({
    'id':    range(50_000),
    'value': np.random.randn(50_000),
    'group': np.random.choice(['A','B','C'],50_000)
})
large_csv.to_csv('/tmp/large.csv', index=False)

chunk_results = []
for chunk in pd.read_csv('/tmp/large.csv', chunksize=10_000):
    result = chunk.groupby('group')['value'].agg(['sum','count'])
    chunk_results.append(result)

final = pd.concat(chunk_results).groupby(level=0).sum()
final['mean'] = final['sum'] / final['count']
print('=== Chunked aggregation ===')
print(final.round(4))

> 💡 **Interview Tip:** `category` dtype stores repeated strings as integer codes internally. A column with 5 unique values across 1M rows as `object` ≈ 50 MB; as `category` ≈ 2–4 MB. Rule of thumb: worth it when unique values < ~50% of total rows.

<a id='21-chaining'></a>
## 21. Method Chaining & Pipelines

Method chaining creates readable, composable, testable data transformations — the functional programming approach to pandas.

In [ ]:
# Imperative style (hard to read, debug)
df = sales.copy()
df['revenue_k'] = df['revenue']/1000
df = df[df['region'].isin(['North','East'])]
df = df[df['revenue_k'] > 1]
df = df.sort_values('revenue_k', ascending=False)
df = df[['order_id','product','region','revenue_k']].reset_index(drop=True).head(5)

# Chained style (clean, readable)
result = (
    sales
    .copy()
    .assign(revenue_k=lambda x: x['revenue']/1000)
    .query("region in ['North','East'] and revenue_k > 1")
    .sort_values('revenue_k', ascending=False)
    [['order_id','product','region','revenue_k']]
    .reset_index(drop=True)
    .head(5)
)
print(result)

In [ ]:
def add_tier(df):
    return df.assign(tier=pd.cut(df['revenue'], bins=[0,500,2000,float('inf')],
                                  labels=['low','mid','high']))

def add_month(df):
    return df.assign(month=df['order_date'].dt.month_name())

def log_shape(df, label=''):
    print(f'[{label}] shape: {df.shape}')
    return df

pipeline_result = (
    sales
    .copy()
    .pipe(log_shape, 'raw')
    .dropna(subset=['discount'])
    .pipe(log_shape, 'after dropna')
    .pipe(add_month)
    .pipe(add_tier)
    .groupby(['month','tier'])['revenue']
    .sum()
    .reset_index()
)
print('\n=== Pipeline result (sample) ===')
print(pipeline_result.head(8))

<a id='22-categorical'></a>
## 22. Categorical Data

`CategoricalDtype` is both a memory optimization and a semantic tool. It enables ordered categories (ordinal), efficient groupby, and correct sort order.

In [ ]:
size_order = pd.CategoricalDtype(categories=['XS','S','M','L','XL','XXL'], ordered=True)

products_cat = pd.DataFrame({
    'product': ['Shirt','Jacket','Pants','Shirt','Jacket'],
    'size':    pd.Categorical(['M','XL','S','L','XS'], dtype=size_order),
    'price':   [29.99, 89.99, 59.99, 29.99, 99.99]
})

print('Items size >= L:')
print(products_cat[products_cat['size'] >= 'L'])

print('\nSorted correctly by size:')
print(products_cat.sort_values('size'))

<a id='23-pitfalls'></a>
## 23. Common Pitfalls & Debugging

These bugs cost engineers hours of debugging. Learn them here, not in production.

In [ ]:
# PITFALL 1: Chained indexing — undefined behavior
df = employees.copy()
# WRONG (may or may not work — implementation detail):
# df[df['dept']=='Engineering']['salary'] = 99999

# CORRECT — single .loc operation:
df.loc[df['dept']=='Engineering','salary'] = 999999
print('Pitfall 1 fixed: always use .loc for assignment')
print(df[df['dept']=='Engineering'][['name','salary']])

In [ ]:
# PITFALL 2: Modifying a slice without .copy()
df = employees.copy()
# WRONG — engineers is a VIEW, modifying it warns/fails:
# engineers = df[df['dept']=='Engineering']
# engineers['salary'] = 0  # SettingWithCopyWarning

# CORRECT:
engineers = df[df['dept']=='Engineering'].copy()
engineers['salary'] = 0  # Only affects the copy
print('Pitfall 2 fixed: .copy() when slicing for mutation')

In [ ]:
# PITFALL 3: NaN comparisons
s = pd.Series([1, 2, np.nan, 4])
print('NaN == NaN:', (s == np.nan).tolist())  # ALL False!
print('isna()    :', s.isna().tolist())        # Correct

# PITFALL 4: Integer columns with NaN become float64
df_int = pd.DataFrame({'a': [1, 2, np.nan, 4]})
print(f'\ndtype with NaN  : {df_int["a"].dtype}')   # float64!
df_int['b'] = pd.array([1,2,pd.NA,4], dtype='Int64')  # Nullable int
print(f'Nullable int    : {df_int["b"].dtype}')
print(df_int)

In [ ]:
# PITFALL 5: groupby silently drops NaN keys
df_null = pd.DataFrame({
    'group': ['A','B',None,'A',None,'B'],
    'value': [10,20,30,40,50,60]
})
print('Default (NaN dropped):')
print(df_null.groupby('group')['value'].sum())

print('\ndropna=False (NaN included):')
print(df_null.groupby('group', dropna=False)['value'].sum())

# PITFALL 6: Row explosion in merge (M:N fanout)
orders_dup = pd.DataFrame({'id':[1,2,2,3],'val':[100,200,300,400]})
tags       = pd.DataFrame({'id':[2,2],'tag':['X','Y']})
merged = pd.merge(orders_dup, tags, on='id', how='left')
print(f'\nBefore merge: {len(orders_dup)} rows')
print(f'After merge : {len(merged)} rows  <- EXPLOSION!')

try:
    pd.merge(orders_dup, tags, on='id', how='left', validate='m:1')
except Exception as e:
    print(f'validate=m:1 caught it: {type(e).__name__}')

<a id='24-interview'></a>
## 24. Interview-Critical Concepts Cheatsheet

The patterns that appear most in data engineering, ML engineering, and analytics interviews.

In [ ]:
# Q1: Find & remove duplicates
df = pd.DataFrame({'a':[1,2,2,3,3,3],'b':['x','y','y','z','z','z']})
print('Duplicate rows:'); print(df[df.duplicated()])
print('\nAfter drop_duplicates:'); print(df.drop_duplicates())
print(f"\nDups in 'a': {df.duplicated(subset=['a']).sum()}")

In [ ]:
# Q2: Running total
t = pd.DataFrame({'date':pd.date_range('2024-01-01',periods=5),'amount':[100,-50,200,-30,150]})
t['balance'] = t['amount'].cumsum()
print('Q2 - Running balance:'); print(t)

# Q3: Lag/Lead features
p = pd.DataFrame({'price':[100,105,103,108,112]})
p['lag1']     = p['price'].shift(1)
p['lead1']    = p['price'].shift(-1)
p['pct_chg']  = p['price'].pct_change()
print('\nQ3 - Lag/Lead:'); print(p.round(4))

# Q4: Top-N per group (classic)
top2 = (
    employees
    .sort_values('salary',ascending=False)
    .groupby('dept')
    .head(2)
    .sort_values(['dept','salary'],ascending=[True,False])
)
print('\nQ4 - Top 2 per dept:'); print(top2[['name','dept','salary']])

In [ ]:
# Q5: Revenue pivot report
pivot_report = (
    sales.groupby(['region','product'])['revenue']
    .sum().unstack(fill_value=0).round(0).astype(int)
)
pivot_report['TOTAL'] = pivot_report.sum(axis=1)
print('Q5 - Pivot report:'); print(pivot_report)

# Q6: Partial string match from list
emails = pd.Series(['alice@gmail.com','bob@company.com','charlie@yahoo.com','diana@company.com'])
corp = ['company.com','corp.org']
print('\nQ6 - Corporate emails:')
print(emails[emails.str.contains('|'.join(corp))])

# Q7: Cross-tabulation
print('\nQ7 - Crosstab product x region:')
print(pd.crosstab(sales['product'], sales['region'], margins=True))

---
# 🏆 FINAL PROJECT
---

<a id='25-project'></a>
## 25. Mini Project: E-Commerce Sales Analysis Pipeline

**Scenario:** You're a data engineer at an e-commerce company. You've received three raw data dumps:
- `orders` — all transactions
- `customers` — customer demographics
- `products` — product catalog

**Tasks:**
1. Ingest and clean all three datasets
2. Build a unified analytics table
3. Compute business KPIs
4. RFM customer segmentation
5. Regional performance report
6. Export to Parquet

In [ ]:
np.random.seed(2024)
N_ORDERS=2000; N_CUST=300; N_PROD=50
cust_ids = [f'C{str(i).zfill(4)}' for i in range(1,N_CUST+1)]
prod_ids = [f'P{str(i).zfill(3)}' for i in range(1,N_PROD+1)]

products_raw = pd.DataFrame({
    'product_id':   prod_ids,
    'product_name': [f'Product_{i}' for i in range(1,N_PROD+1)],
    'category':     np.random.choice(['Electronics','Clothing','Home','Sports','Books'],N_PROD),
    'unit_cost':    np.random.uniform(10,800,N_PROD).round(2),
})
products_raw['list_price'] = (products_raw['unit_cost']*np.random.uniform(1.3,2.5,N_PROD)).round(2)

customers_raw = pd.DataFrame({
    'customer_id': cust_ids,
    'name':        [f'Customer_{i}' for i in range(1,N_CUST+1)],
    'region':      np.random.choice(['North','South','East','West'],N_CUST),
    'age':         np.random.randint(18,70,N_CUST).astype(float),
    'signup_date': pd.date_range('2020-01-01',periods=N_CUST,freq='3D').strftime('%Y-%m-%d'),
    'tier':        np.random.choice(['bronze','silver','gold','platinum'],N_CUST,p=[0.5,0.3,0.15,0.05]),
})
null_idx = np.random.choice(N_CUST,20,replace=False)
customers_raw.loc[null_idx,'age'] = np.nan

orders_raw = pd.DataFrame({
    'order_id':    [f'ORD{str(i).zfill(6)}' for i in range(1,N_ORDERS+1)],
    'customer_id': np.random.choice(cust_ids,N_ORDERS),
    'product_id':  np.random.choice(prod_ids,N_ORDERS),
    'quantity':    np.random.randint(1,8,N_ORDERS),
    'order_date':  pd.date_range('2023-01-01','2023-12-31',periods=N_ORDERS).strftime('%Y-%m-%d'),
    'status':      np.random.choice(['completed','returned','pending'],N_ORDERS,p=[0.85,0.10,0.05]),
    'discount_pct':np.random.choice([0,5,10,15,20,np.nan],N_ORDERS,p=[0.4,0.2,0.2,0.1,0.05,0.05]),
})
print(f'Generated: {len(orders_raw)} orders, {len(customers_raw)} customers, {len(products_raw)} products')

In [ ]:
def clean_orders(df):
    return (
        df.copy()
        .assign(
            order_date   = lambda x: pd.to_datetime(x['order_date']),
            discount_pct = lambda x: x['discount_pct'].fillna(0).astype(float),
            quantity     = lambda x: x['quantity'].astype('int16'),
        )
        .query("status != 'returned'")
    )

def clean_customers(df):
    tier_cat = pd.CategoricalDtype(['bronze','silver','gold','platinum'], ordered=True)
    return (
        df.copy()
        .assign(
            signup_date = lambda x: pd.to_datetime(x['signup_date']),
            age         = lambda x: x['age'].fillna(x['age'].median()).astype('int8'),
            tier        = lambda x: x['tier'].astype(tier_cat),
        )
    )

def clean_products(df):
    return (
        df.copy()
        .assign(
            category = lambda x: x['category'].astype('category'),
            margin   = lambda x: ((x['list_price']-x['unit_cost'])/x['list_price']*100).round(1)
        )
    )

orders_c    = clean_orders(orders_raw)
customers_c = clean_customers(customers_raw)
products_c  = clean_products(products_raw)
print(f'Clean orders: {len(orders_c)}')
print(orders_c.dtypes)

In [ ]:
analytics = (
    orders_c
    .merge(customers_c[['customer_id','name','region','tier','age']], on='customer_id', how='left')
    .merge(products_c[['product_id','product_name','category','list_price','unit_cost','margin']],
           on='product_id', how='left')
    .assign(
        gross_revenue = lambda x: x['quantity'] * x['list_price'],
        discount_amt  = lambda x: x['gross_revenue'] * x['discount_pct']/100,
        net_revenue   = lambda x: x['gross_revenue'] - x['discount_amt'],
        cogs          = lambda x: x['quantity'] * x['unit_cost'],
        gross_profit  = lambda x: (x['gross_revenue'] - x['discount_amt']) - x['quantity'] * x['unit_cost'],
        month         = lambda x: x['order_date'].dt.to_period('M'),
        quarter       = lambda x: x['order_date'].dt.quarter,
    )
    .round({'gross_revenue':2,'net_revenue':2,'gross_profit':2})
)
print(f'Analytics table shape: {analytics.shape}')
print(analytics[['order_id','region','category','quantity','net_revenue','gross_profit']].head(5))

In [ ]:
total_rev   = analytics['net_revenue'].sum()
total_profit= analytics['gross_profit'].sum()
margin_pct  = total_profit/total_rev*100
n_orders    = len(analytics)
n_customers = analytics['customer_id'].nunique()

print('='*50)
print('        ANNUAL BUSINESS KPIs')
print('='*50)
print(f'  Total Revenue       : ${total_rev:>12,.0f}')
print(f'  Gross Profit        : ${total_profit:>12,.0f}')
print(f'  Gross Margin        : {margin_pct:>11.1f}%')
print(f'  Total Orders        : {n_orders:>13,}')
print(f'  Unique Customers    : {n_customers:>13,}')
print(f'  Avg Order Value     : ${analytics["net_revenue"].mean():>12,.2f}')
print('='*50)

In [ ]:
reference = pd.Timestamp('2024-01-01')

rfm = (
    analytics
    .groupby('customer_id')
    .agg(
        recency   = ('order_date',  lambda x: (reference - x.max()).days),
        frequency = ('order_id',    'count'),
        monetary  = ('net_revenue', 'sum')
    )
    .assign(
        r_score = lambda x: pd.qcut(x['recency'],  q=5, labels=[5,4,3,2,1]).astype(int),
        f_score = lambda x: pd.qcut(x['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int),
        m_score = lambda x: pd.qcut(x['monetary'],  q=5, labels=[1,2,3,4,5]).astype(int),
    )
    .assign(rfm_score=lambda x: x['r_score']+x['f_score']+x['m_score'])
)
rfm['segment'] = pd.cut(rfm['rfm_score'], bins=[0,6,9,12,15],
                         labels=['At Risk','Needs Attention','Loyal','Champions'])

print('=== RFM Segment Distribution ===')
print(rfm['segment'].value_counts())
print('\n=== Top 5 Champions ===')
print(rfm[rfm['segment']=='Champions']
      .sort_values('rfm_score',ascending=False)
      .head(5)[['recency','frequency','monetary','rfm_score']].round(2))

In [ ]:
regional = (
    analytics
    .groupby('region')
    .agg(
        total_revenue    = ('net_revenue',  'sum'),
        total_profit     = ('gross_profit', 'sum'),
        order_count      = ('order_id',     'count'),
        unique_customers = ('customer_id',  'nunique'),
        avg_order_value  = ('net_revenue',  'mean'),
    )
    .assign(
        margin_pct          = lambda x: (x['total_profit']/x['total_revenue']*100).round(1),
        revenue_share_pct   = lambda x: (x['total_revenue']/x['total_revenue'].sum()*100).round(1),
        orders_per_customer = lambda x: (x['order_count']/x['unique_customers']).round(2),
    )
    .sort_values('total_revenue', ascending=False)
    .round({'total_revenue':0,'total_profit':0,'avg_order_value':2})
)
print('=== Regional Performance Report ===')
print(regional.to_string())

In [ ]:
monthly = (
    analytics
    .groupby('month')
    .agg(revenue=('net_revenue','sum'), profit=('gross_profit','sum'), orders=('order_id','count'))
    .assign(
        rev_mom_chg = lambda x: x['revenue'].pct_change()*100,
        margin      = lambda x: (x['profit']/x['revenue']*100).round(1),
        cumrev      = lambda x: x['revenue'].cumsum()
    )
    .round(1)
)
print('=== Monthly Trend ==='); print(monthly.to_string())

In [ ]:
analytics.to_parquet('/tmp/analytics_table.parquet', index=False)
rfm.reset_index().to_csv('/tmp/rfm_segments.csv', index=False)
regional.to_csv('/tmp/regional_performance.csv')
monthly.to_csv('/tmp/monthly_trend.csv')

print('=== Exports ===')
for f in ['/tmp/analytics_table.parquet','/tmp/rfm_segments.csv',
          '/tmp/regional_performance.csv','/tmp/monthly_trend.csv']:
    print(f'  {f:45s} {os.path.getsize(f):>10,} bytes')
print('\n Pipeline complete!')

### 🏋️ Challenge Extensions

1. **Category Deep Dive:** Which category has the highest margin AND fastest month-over-month growth?
2. **Cohort Analysis:** Group customers by signup quarter and compare retention (repeat purchase rate) across cohorts.
3. **Anomaly Detection:** Use rolling statistics to flag orders where revenue deviates > 3σ from the 30-day rolling mean.
4. **Performance Audit:** Replace all `.apply()` calls in your code with vectorized alternatives and benchmark the speedup.


---
## 📋 Quick Reference Cheatsheet

```python
# CREATION
pd.DataFrame([{'a':1},{'a':2}])           # from list of dicts
pd.read_csv('f.csv', dtype={}, parse_dates=['d'])
pd.read_parquet('f.parquet')

# INSPECTION
df.info() / df.describe() / df.head()
df.isna().sum() / df.nunique()

# INDEXING
df.loc[label_row, label_col]   # label, inclusive
df.iloc[int_row, int_col]      # position, exclusive end
df.at[r,c] / df.iat[r,c]      # fast scalar

# FILTERING
df[mask] / df.query('col > val')
df[df['c'].isin(lst)] / df[df['c'].between(a,b)]

# MUTATING
df.assign(col=lambda x: ...)   # immutable (preferred)
df.loc[mask, col] = value
df.drop(columns=['x'])

# GROUPBY
df.groupby('c').agg(res=('val','sum'))
df.groupby('c')['v'].transform('mean')  # same-shape output

# MERGING
pd.merge(L, R, on='k', how='left', validate='m:1')
pd.concat([df1, df2], ignore_index=True)

# RESHAPING
df.pivot_table(values, index, columns, aggfunc)
pd.melt(df, id_vars=['id'], var_name='k', value_name='v')

# TIME SERIES
df['ts'].dt.year / .month / .day_name()
df.resample('ME')['v'].sum()
df['v'].rolling(7).mean()
df['v'].ewm(span=12).mean()

# PERFORMANCE
df['c'].astype('category')                  # low-cardinality strings
pd.to_numeric(s, downcast='integer')        # smaller int types
pd.read_csv('f', chunksize=10_000)          # streaming large files
np.select(conditions, choices, default)     # vectorized if-else
```

---
*Happy data engineering! 🐼*